# Build a Lakeflow Spark Declarative Pipeline with three tables and prove expectations fire against a deliberately bad file

The brittle notebook job has bit the team three times in a quarter and the pipeline lead is done with `.dropna()`-and-pray. The audit team wants a pipeline that fails loudly on a primary-key violation, not one that silently drops the bad row and ships a wrong daily total to the CFO. You have one session to rewrite the job as a Lakeflow Spark Declarative Pipeline with explicit expectations, run it clean, then deliberately break it to prove the failure mode is loud and observable.

The hands-on work:

- Create a UC schema `workspace.default.labrun_decl_pipeline` and a UC Volume `source_files` for the JSON inputs.
- Drop three clean JSON files (one order per file) into the volume.
- Author `pipeline.py` with three `@dlt.table` functions: `bronze_orders` (raw ingest), `silver_orders` (cleaned with expectations), `gold_orders_daily` (aggregated daily totals).
- Attach `@dlt.expect_or_drop` on null `customer_id` and `@dlt.expect_or_fail` on null `order_id` to silver.
- Create the pipeline via the Lakeflow Pipelines API and trigger a clean run. Confirm bronze=3, silver=3, gold=1.
- Drop a deliberately bad fourth file (30% null order_id) into the volume, re-trigger the pipeline, and confirm the run fails at the expectation with the violation visible in the event log.

**Time.** Plan for about 70 minutes hands-on. Lakeflow Spark Declarative Pipelines on Free Edition have a 30 to 90 second cold start; two runs eat the bulk of your wall-clock time. Budget up to 120 minutes for the session window.

**Cost.** Zero dollars. Two pipeline runs on serverless against tiny inputs. The daily compute quota is the only thing this lab touches.

This lab maps to Databricks DE Associate Domain 3 (Transformation, ~31%) and Domain 4 (Lakeflow Jobs and pipelines, ~10%), both provisional.

**Rename callout.** If your other prep material says "Delta Live Tables" or "DLT" it means Lakeflow Spark Declarative Pipelines on the current exam. The Python decorator import (`import dlt`) is retained for compatibility, but the product surface, UI navigation, and exam terminology are now Lakeflow Spark Declarative Pipelines. If it says "Databricks Repos" it means Databricks Git Folders.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks and the Databricks SDK. Pinned versions per
# LAB-CREATION-BLUEPRINT.md build rules.

!pip install --quiet labrun-checks==0.6.0 databricks-sdk==0.40.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Pipeline definition and event-log polling
# both go through the Databricks SDK; the pipeline source file lives in a
# workspace path we create via the SDK as well (no Git Folder dependency
# for the Free Edition path because Git Folders require a connected repo).

import atexit
import getpass
import io
import json
import sys
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState
from databricks.sdk.service import pipelines as p_svc
from databricks.sdk.service import workspace as ws_svc

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupResource,
)

LAB_ID = "lakeflow-declarative-pipeline-with-expectations"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_decl_pipeline"
LAB_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"

SOURCE_VOLUME = "source_files"
SOURCE_VOLUME_FQN = f"{LAB_SCHEMA_FQN}.{SOURCE_VOLUME}"
SOURCE_VOLUME_PATH = f"/Volumes/{CATALOG}/{PARENT_SCHEMA}/{LAB_SCHEMA}/{SOURCE_VOLUME}"

BRONZE_TABLE = "bronze_orders"
SILVER_TABLE = "silver_orders"
GOLD_TABLE = "gold_orders_daily"
BRONZE_FQN = f"{LAB_SCHEMA_FQN}.{BRONZE_TABLE}"
SILVER_FQN = f"{LAB_SCHEMA_FQN}.{SILVER_TABLE}"
GOLD_FQN = f"{LAB_SCHEMA_FQN}.{GOLD_TABLE}"

PIPELINE_NAME = f"labrun-decl-pipeline-with-expectations"
PIPELINE_ID = None  # set in Task 2 after pipeline create
PIPELINE_SOURCE_DIR = None  # set in Task 2 after workspace path is created
PIPELINE_SOURCE_PATH = None  # set in Task 2

# Pipeline source as a single Python file. The decorator namespace is still
# `dlt` even though the product is Lakeflow Spark Declarative Pipelines.
PIPELINE_SOURCE = f'''import dlt
from pyspark.sql import functions as F


SOURCE_PATH = "{SOURCE_VOLUME_PATH}"


@dlt.table(name="{BRONZE_TABLE}", comment="Raw orders ingest from UC Volume.")
def bronze_orders():
    return (
        spark.read.format("json")
        .option("multiLine", "false")
        .load(SOURCE_PATH + "/*.json")
    )


@dlt.table(
    name="{SILVER_TABLE}",
    comment="Cleaned orders with PK and null-customer expectations.",
)
@dlt.expect_or_drop("non_null_customer_id", "customer_id IS NOT NULL")
@dlt.expect_or_fail("non_null_order_id", "order_id IS NOT NULL")
def silver_orders():
    return (
        spark.read.table("LIVE.{BRONZE_TABLE}")
        .withColumn("order_date", F.to_date(F.col("order_ts")))
    )


@dlt.table(
    name="{GOLD_TABLE}",
    comment="Daily order count and total revenue.",
)
def gold_orders_daily():
    return (
        spark.read.table("LIVE.{SILVER_TABLE}")
        .groupBy("order_date")
        .agg(
            F.count("*").alias("order_count"),
            F.sum("order_total").alias("revenue_usd"),
        )
    )
'''

CLEAN_ORDERS = [
    {{"order_id": "O-001", "customer_id": "C-1", "order_total": 49.99, "order_ts": "2026-05-13T10:00:00"}},
    {{"order_id": "O-002", "customer_id": "C-2", "order_total": 19.95, "order_ts": "2026-05-13T11:15:30"}},
    {{"order_id": "O-003", "customer_id": "C-3", "order_total": 75.50, "order_ts": "2026-05-13T13:45:12"}},
]

BAD_ORDERS = [
    {{"order_id": None,    "customer_id": "C-4", "order_total": 30.00, "order_ts": "2026-05-13T14:00:00"}},
    {{"order_id": "O-005", "customer_id": "C-5", "order_total": 12.50, "order_ts": "2026-05-13T14:05:00"}},
    {{"order_id": None,    "customer_id": "C-6", "order_total": 99.00, "order_ts": "2026-05-13T14:10:00"}},
    {{"order_id": "O-007", "customer_id": "C-7", "order_total": 18.75, "order_ts": "2026-05-13T14:15:00"}},
    {{"order_id": "O-008", "customer_id": "C-8", "order_total": 22.00, "order_ts": "2026-05-13T14:20:00"}},
]

In [ ]:
# NBVAL_SKIP
# Register the session, validate Databricks credentials via the SDK, resolve
# the Starter SQL warehouse for orphan scans, and expose a run_sql() helper.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL: ").strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired:")
    print(f"  {exc}")
    raise SystemExit(1)
CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found. Recreate the Starter warehouse and re-run.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180):
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid, statement=statement, wait_timeout="50s",
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement}")
        if time.time() > deadline:
            raise TimeoutError(f"SQL did not finish in {wait_seconds}s. Last state: {state}.")
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)
    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    return {"columns": columns, "rows": rows}


# Set the workspace pipeline source path now that we know CALLER_USER.
PIPELINE_SOURCE_DIR = f"/Workspace/Users/{CALLER_USER}/labrun-decl-pipeline-with-expectations"
PIPELINE_SOURCE_PATH = f"{PIPELINE_SOURCE_DIR}/pipeline.py"
print(f"Pipeline source path will be: {PIPELINE_SOURCE_PATH}")

register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan. The Lakeflow
# pipeline is critical=True because a running pipeline consumes serverless
# DBUs against the daily quota. The lakeflow_pipeline adapter stops the
# pipeline before deleting; the stop API is idempotent.


def _find_pipelines_by_name(name):
    found = []
    try:
        for pipeline in w.pipelines.list_pipelines(filter=f"name LIKE '{name}'"):
            if pipeline.name == name:
                found.append(pipeline)
    except Exception:
        pass
    return found


CLEANUP_MANIFEST = [
    CleanupResource(
        type="lakeflow_pipeline",
        id=PIPELINE_NAME,
        region="databricks",
        critical=True,
        cli_delete_command=f"databricks pipelines stop --name {PIPELINE_NAME} && databricks pipelines delete --name {PIPELINE_NAME}",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=GOLD_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {GOLD_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=SILVER_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {SILVER_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=BRONZE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {BRONZE_FQN}\"",
    ),
    CleanupResource(
        type="uc_volume_contents",
        id=SOURCE_VOLUME_FQN,
        region="databricks",
        cli_delete_command=f"databricks fs rm -r dbfs:{SOURCE_VOLUME_PATH}/",
    ),
    CleanupResource(
        type="uc_volume",
        id=SOURCE_VOLUME_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP VOLUME IF EXISTS {SOURCE_VOLUME_FQN}\"",
    ),
    CleanupResource(
        type="workspace_path",
        id="<set in Task 2>",  # rewritten below once PIPELINE_SOURCE_DIR is known
        region="databricks",
        cli_delete_command="databricks workspace delete --recursive <set in Task 2>",
    ),
    CleanupResource(
        type="uc_schema",
        id=LAB_SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {LAB_SCHEMA_FQN} CASCADE\"",
    ),
]
# Patch the workspace_path entry now that PIPELINE_SOURCE_DIR is known.
for res in CLEANUP_MANIFEST:
    if res.type == "workspace_path":
        res.id = PIPELINE_SOURCE_DIR
        res.cli_delete_command = f"databricks workspace delete --recursive {PIPELINE_SOURCE_DIR}"


class DatabricksCleanupAdapter:
    def delete_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_volume":
            run_sql(f"DROP VOLUME IF EXISTS {rid}")
        elif rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
            except Exception:
                return
            for entry in listing:
                try:
                    if entry.is_directory:
                        w.files.delete_directory(entry.path)
                    else:
                        w.files.delete(entry.path)
                except Exception:
                    pass
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        elif rtype == "workspace_path":
            try:
                w.workspace.delete(rid, recursive=True)
            except Exception:
                pass
        elif rtype == "lakeflow_pipeline":
            for pipeline in _find_pipelines_by_name(rid):
                try:
                    w.pipelines.stop(pipeline.pipeline_id)
                except Exception:
                    pass
                deadline = time.time() + 120
                while time.time() < deadline:
                    try:
                        info = w.pipelines.get(pipeline.pipeline_id)
                        state = (info.state.value if info.state else "").upper()
                        if state in ("IDLE", "STOPPED", "FAILED", ""):
                            break
                    except Exception:
                        break
                    time.sleep(3)
                try:
                    w.pipelines.delete(pipeline.pipeline_id)
                except Exception:
                    pass
        else:
            raise ValueError(f"Unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "uc_managed_table":
            catalog, schema, table = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                f"AND table_name = '{table}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        if rtype == "uc_volume":
            catalog, schema, volume = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.volumes "
                f"WHERE volume_catalog = '{catalog}' AND volume_schema = '{schema}' "
                f"AND volume_name = '{volume}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        if rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                return len(list(w.files.list_directory_contents(volume_path))) > 0
            except Exception:
                return False
        if rtype == "uc_schema":
            catalog, schema = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.schemata "
                f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        if rtype == "workspace_path":
            try:
                w.workspace.get_status(rid)
                return True
            except Exception:
                return False
        if rtype == "lakeflow_pipeline":
            return len(_find_pipelines_by_name(rid)) > 0
        return False

    def tag_scan(self, credentials, lab_slug, region):
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
            (
                "SELECT catalog_name, schema_name, volume_name FROM system.information_schema.volume_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql)
            except Exception:
                continue
            for row in result["rows"]:
                arns.append(fmt(row))
        # Tag-scan pipelines by configuration key (pipelines do not have first-class tags).
        try:
            for pipeline in w.pipelines.list_pipelines():
                info = w.pipelines.get(pipeline.pipeline_id)
                config = (info.spec.configuration if info.spec else None) or {}
                if config.get(LAB_TAG_KEY) == lab_slug:
                    arns.append(f"pipeline:{pipeline.name}")
        except Exception:
            pass
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


orphans = CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")
if orphans:
    print(f"BLOCKED: existing objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Stand up the schema, the volume, and three clean JSON files

The pipeline needs a source it can read. Two managed objects, three files:

1. `CREATE SCHEMA workspace.default.labrun_decl_pipeline` and tag it.
2. `CREATE VOLUME workspace.default.labrun_decl_pipeline.source_files` and tag it.
3. Upload three JSON files (one order per file) under the volume. The files come from the `CLEAN_ORDERS` Python list defined in the imports cell.

The pipeline source reads `*.json` from the volume root, so file names matter only in that they are unique.

In [ ]:
# NBVAL_SKIP
# Task 1: Schema, volume, three clean JSON files.

# YOUR CODE: Run CREATE SCHEMA IF NOT EXISTS LAB_SCHEMA_FQN via run_sql().

# YOUR CODE: Tag the schema with ALTER SCHEMA LAB_SCHEMA_FQN SET TAGS
# ('labrun_lab_slug' = LAB_TAG_VALUE).

# YOUR CODE: Run CREATE VOLUME IF NOT EXISTS SOURCE_VOLUME_FQN via run_sql().

# YOUR CODE: Tag the volume with ALTER VOLUME SOURCE_VOLUME_FQN SET TAGS.

for i, order in enumerate(CLEAN_ORDERS, start=1):
    file_path = f"{SOURCE_VOLUME_PATH}/clean-{i:03d}.json"
    payload = (json.dumps(order) + "\n").encode("utf-8")
    # YOUR CODE: Upload payload to file_path via
    # w.files.upload(file_path=file_path, contents=io.BytesIO(payload),
    # overwrite=True).
    print(f"Uploaded {file_path}")

listing = list(w.files.list_directory_contents(SOURCE_VOLUME_PATH + "/"))
print(f"{len(listing)} files staged in source volume")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Schema tagged, volume tagged, three clean JSON files staged.


def checkpoint_1(session):
    try:
        schema_sql = (
            "SELECT 1 FROM system.information_schema.schemata "
            f"WHERE catalog_name = '{CATALOG}' AND schema_name = '{LAB_SCHEMA}'"
        )
        if not run_sql(schema_sql)["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=f"Schema {LAB_SCHEMA_FQN} not found.",
            )
        tag_sql = (
            "SELECT tag_value FROM system.information_schema.schema_tags "
            f"WHERE catalog_name = '{CATALOG}' AND schema_name = '{LAB_SCHEMA}' "
            f"AND tag_name = '{LAB_TAG_KEY}'"
        )
        if not any(r[0] == LAB_TAG_VALUE for r in run_sql(tag_sql)["rows"]):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Schema is missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. "
                    f"Apply ALTER SCHEMA SET TAGS."
                ),
            )

        vol_sql = (
            "SELECT 1 FROM system.information_schema.volumes "
            f"WHERE volume_catalog = '{CATALOG}' AND volume_schema = '{LAB_SCHEMA}' "
            f"AND volume_name = '{SOURCE_VOLUME}'"
        )
        if not run_sql(vol_sql)["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=f"Volume {SOURCE_VOLUME_FQN} not found.",
            )

        try:
            listing = list(w.files.list_directory_contents(SOURCE_VOLUME_PATH + "/"))
        except Exception as exc:
            return CheckpointResult(
                status="fail",
                error_reason=f"Could not list {SOURCE_VOLUME_PATH}/: {exc}",
            )
        json_files = [e for e in listing if e.path.endswith(".json")]
        if len(json_files) < 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Source volume has {len(json_files)} *.json files; expected 3. "
                    f"Re-run the upload loop."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint message names what is missing: schema row, schema tag, volume row, or file count. Each is a different SQL statement or upload call.

</details>

<details><summary>Hint 2 (direction)</summary>

Four SQL statements (schema, schema tag, volume, volume tag) plus three uploads. The volume is a managed volume, no `LOCATION` clause. Uploads go to `/Volumes/workspace/default/labrun_decl_pipeline/source_files/clean-NNN.json`. Use the `w.files.upload(file_path=..., contents=io.BytesIO(payload), overwrite=True)` SDK call.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
run_sql(f"CREATE SCHEMA IF NOT EXISTS {LAB_SCHEMA_FQN}")
run_sql(f"ALTER SCHEMA {LAB_SCHEMA_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")
run_sql(f"CREATE VOLUME IF NOT EXISTS {SOURCE_VOLUME_FQN}")
run_sql(f"ALTER VOLUME {SOURCE_VOLUME_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

for i, order in enumerate(CLEAN_ORDERS, start=1):
    file_path = f"{SOURCE_VOLUME_PATH}/clean-{i:03d}.json"
    payload = (json.dumps(order) + "\n").encode("utf-8")
    w.files.upload(file_path=file_path, contents=io.BytesIO(payload), overwrite=True)
```

</details>

**Wallet check.** Still at $0.00. Three tiny JSON files in a managed UC Volume. The pipeline has not run yet, so no serverless DBUs.

## Task 2: Write `pipeline.py` to a workspace path and create the pipeline

The pipeline source file is the `PIPELINE_SOURCE` Python string from the imports cell. It defines three `@dlt.table` functions and attaches two expectations to silver: `expect_or_drop` on null `customer_id`, `expect_or_fail` on null `order_id`. Save it to `/Workspace/Users/<your-user>/labrun-decl-pipeline-with-expectations/pipeline.py`, then create the Lakeflow Spark Declarative Pipeline via the SDK.

Pipeline configuration matters:

- `name=PIPELINE_NAME`
- `catalog="workspace"`, `target="default.labrun_decl_pipeline"` so the pipeline writes the three tables under our lab schema
- `libraries=[{"file": {"path": pipeline.py path}}]`
- `configuration={"labrun_lab_slug": LAB_TAG_VALUE}` so the orphan scan can find the pipeline later
- `serverless=True` because Free Edition is serverless-only
- `continuous=False` because Free Edition does not support continuous mode

Trigger the pipeline once with `full_refresh=False`. Poll the events API until the run reaches `COMPLETED` or `FAILED`. The cold-start budget is 300 seconds.

In [ ]:
# NBVAL_SKIP
# Task 2: Write pipeline.py, create the pipeline, trigger the clean run.

# Ensure the workspace folder exists, then write pipeline.py.
try:
    w.workspace.mkdirs(PIPELINE_SOURCE_DIR)
except Exception as exc:
    print(f"mkdirs warning (often safe to ignore): {exc}")

# YOUR CODE: Import the workspace ImportFormat enum and call
# w.workspace.import_(path=PIPELINE_SOURCE_PATH, format=ImportFormat.SOURCE,
# language=Language.PYTHON, content=base64.b64encode(PIPELINE_SOURCE.encode()).decode(),
# overwrite=True). The Workspace API takes content as base64-encoded bytes.

import base64
from databricks.sdk.service.workspace import ImportFormat, Language

# YOUR CODE: Use w.workspace.import_ to upload PIPELINE_SOURCE as a Python
# SOURCE file at PIPELINE_SOURCE_PATH. content = base64.b64encode(
# PIPELINE_SOURCE.encode("utf-8")).decode("utf-8"). Pass overwrite=True.

print(f"Wrote {PIPELINE_SOURCE_PATH}")

# YOUR CODE: Create the pipeline via w.pipelines.create(...). Required
# arguments: name=PIPELINE_NAME, catalog=CATALOG,
# target=f"{PARENT_SCHEMA}.{LAB_SCHEMA}",
# libraries=[p_svc.PipelineLibrary(file=p_svc.FileLibrary(path=PIPELINE_SOURCE_PATH))],
# configuration={LAB_TAG_KEY: LAB_TAG_VALUE},
# serverless=True, continuous=False, channel="CURRENT".
# Assign the returned pipeline_id to PIPELINE_ID via the global declaration.

global PIPELINE_ID
# YOUR CODE: PIPELINE_ID = w.pipelines.create(...).pipeline_id

print(f"Pipeline created: {PIPELINE_NAME} (id={PIPELINE_ID})")

# Trigger the clean run.
update = w.pipelines.start_update(PIPELINE_ID, full_refresh=False)
update_id = update.update_id
print(f"Started update {update_id}. Asking the pipeline to put on its reading glasses...")

deadline = time.time() + 300
final_state = None
while time.time() < deadline:
    info = w.pipelines.get_update(PIPELINE_ID, update_id)
    state = (info.update.state.value if info.update and info.update.state else "")
    state = state.upper() if state else ""
    if state in ("COMPLETED", "FAILED", "CANCELED"):
        final_state = state
        break
    time.sleep(8)

print(f"Update final state: {final_state}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Pipeline exists with the right libraries and target.
# Clean run COMPLETED. Bronze=3, silver=3, gold=1.


def checkpoint_2(session):
    try:
        if PIPELINE_ID is None:
            return CheckpointResult(
                status="fail",
                error_reason="PIPELINE_ID is None. Did w.pipelines.create() run?",
            )
        info = w.pipelines.get(PIPELINE_ID)
        if not info or not info.spec:
            return CheckpointResult(
                status="fail",
                error_reason=f"Could not fetch pipeline {PIPELINE_ID}.",
            )

        libraries = info.spec.libraries or []
        if len(libraries) != 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Pipeline has {len(libraries)} libraries; expected exactly 1 "
                    f"(the pipeline.py file)."
                ),
            )
        lib = libraries[0]
        lib_path = ""
        if lib.file and lib.file.path:
            lib_path = lib.file.path
        elif lib.notebook and lib.notebook.path:
            lib_path = lib.notebook.path
        if PIPELINE_SOURCE_PATH not in lib_path:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Pipeline library path {lib_path!r} does not match expected "
                    f"{PIPELINE_SOURCE_PATH}."
                ),
            )

        catalog = info.spec.catalog or ""
        target = info.spec.target or ""
        # Lakeflow on UC accepts target as `schema` or `parent.schema` depending
        # on engine version. Accept either shape here.
        target_ok = (
            catalog == CATALOG
            and (LAB_SCHEMA in target)
        )
        if not target_ok:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Pipeline catalog={catalog!r} target={target!r}; expected "
                    f"catalog={CATALOG!r} and target containing {LAB_SCHEMA!r}."
                ),
            )

        config = info.spec.configuration or {}
        if config.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Pipeline configuration missing key {LAB_TAG_KEY}={LAB_TAG_VALUE}. "
                    f"Set it in the create() configuration dict so the orphan scan "
                    f"can find this pipeline."
                ),
            )

        # Row counts.
        for fqn, expected, label in [
            (BRONZE_FQN, 3, "bronze"),
            (SILVER_FQN, 3, "silver"),
            (GOLD_FQN, 1, "gold"),
        ]:
            count_rows = run_sql(f"SELECT count(*) FROM {fqn}")["rows"]
            actual = int(count_rows[0][0]) if count_rows else 0
            if actual != expected:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{fqn} has {actual} rows; expected {expected} after the "
                        f"clean pipeline run. Confirm the {label} table materialized."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint either complains that the pipeline does not exist, that the libraries / target / configuration do not match, or that the row counts are off after the clean run. The error message names which check failed.

</details>

<details><summary>Hint 2 (direction)</summary>

Two SDK calls. First, `w.workspace.import_(path=PIPELINE_SOURCE_PATH, format=ImportFormat.SOURCE, language=Language.PYTHON, content=base64.b64encode(PIPELINE_SOURCE.encode()).decode(), overwrite=True)`. Second, `PIPELINE_ID = w.pipelines.create(name=PIPELINE_NAME, catalog=CATALOG, target=f"{PARENT_SCHEMA}.{LAB_SCHEMA}", libraries=[p_svc.PipelineLibrary(file=p_svc.FileLibrary(path=PIPELINE_SOURCE_PATH))], configuration={LAB_TAG_KEY: LAB_TAG_VALUE}, serverless=True, continuous=False, channel="CURRENT").pipeline_id`. Then start an update and poll get_update.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
import base64
from databricks.sdk.service.workspace import ImportFormat, Language

w.workspace.import_(
    path=PIPELINE_SOURCE_PATH,
    format=ImportFormat.SOURCE,
    language=Language.PYTHON,
    content=base64.b64encode(PIPELINE_SOURCE.encode("utf-8")).decode("utf-8"),
    overwrite=True,
)

global PIPELINE_ID
PIPELINE_ID = w.pipelines.create(
    name=PIPELINE_NAME,
    catalog=CATALOG,
    target=f"{PARENT_SCHEMA}.{LAB_SCHEMA}",
    libraries=[p_svc.PipelineLibrary(file=p_svc.FileLibrary(path=PIPELINE_SOURCE_PATH))],
    configuration={LAB_TAG_KEY: LAB_TAG_VALUE},
    serverless=True,
    continuous=False,
    channel="CURRENT",
).pipeline_id
```

</details>

**Wallet check.** Still at $0.00. One pipeline run on serverless against three rows. Free Edition swallowed it; the daily quota took a sip.

## Task 3: Drop the bad file and prove the expectation fires loudly

The `BAD_ORDERS` list (from the imports cell) has 5 rows where two have null `order_id`. Drop them as a single JSON file (one order per line, ndjson) into the same volume:

1. Upload `BAD_ORDERS` as `bad-orders.json` with one JSON object per line.
2. Trigger the pipeline again with `full_refresh=False`.
3. Poll until the update reaches FAILED. The expected failure is `non_null_order_id` (the `expect_or_fail` expectation) violating on the two null rows.

After the failed run, query the pipeline event log (`w.pipelines.list_pipeline_events`) and confirm at least one event references the expectation failure. The checkpoint inspects the events list for an event whose message or details surface the expectation name.

In [ ]:
# NBVAL_SKIP
# Task 3: Upload the bad file, re-trigger, confirm the FAIL.

# YOUR CODE: Build an ndjson payload from BAD_ORDERS (one JSON per line)
# and upload it to f"{SOURCE_VOLUME_PATH}/bad-orders.json" via w.files.upload.

print(f"Bad file staged.")

# YOUR CODE: Start a new update via w.pipelines.start_update(PIPELINE_ID,
# full_refresh=False) and capture update.update_id.

# YOUR CODE: Poll w.pipelines.get_update(PIPELINE_ID, update_id) until
# the state is COMPLETED, FAILED, or CANCELED. Print the final state.
# Use a 300-second deadline.

# Expected: final_state == FAILED, because the @dlt.expect_or_fail on
# non_null_order_id violated against the two null rows in BAD_ORDERS.
print("Expected outcome: state=FAILED with expectation violation visible in events.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Most recent pipeline update is FAILED; the pipeline events
# list contains an event whose message references the failing expectation.


def checkpoint_3(session):
    try:
        if PIPELINE_ID is None:
            return CheckpointResult(
                status="fail",
                error_reason="PIPELINE_ID is None.",
            )

        updates = list(w.pipelines.list_updates(PIPELINE_ID).updates or [])
        if len(updates) < 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Pipeline has {len(updates)} updates; expected at least 2 "
                    f"(the clean run plus the bad-file run). Trigger the bad-file "
                    f"update."
                ),
            )
        latest = updates[0]
        state_value = (latest.state.value if latest.state else "") or ""
        if state_value.upper() != "FAILED":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Most recent update state is {state_value!r}; expected FAILED. "
                    f"Did the bad-orders.json upload land before the trigger?"
                ),
            )

        # Walk recent events looking for the expectation failure.
        events = list(
            w.pipelines.list_pipeline_events(PIPELINE_ID, max_results=200) or []
        )
        expectation_hit = False
        for ev in events:
            blob = ""
            if ev.message:
                blob += ev.message + " "
            if ev.details:
                try:
                    blob += json.dumps(ev.details.as_dict()) if hasattr(ev.details, "as_dict") else str(ev.details)
                except Exception:
                    blob += str(ev.details)
            blob = blob.lower()
            if "expect" in blob and ("non_null_order_id" in blob or "fail" in blob):
                expectation_hit = True
                break
        if not expectation_hit:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Pipeline events do not surface an expectation failure for "
                    "non_null_order_id. The run failed but for a different reason; "
                    "confirm the bad file actually contains null order_id rows and "
                    "that silver_orders has the @dlt.expect_or_fail decorator."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Two failure shapes. Either the run is not FAILED (the bad file did not land or the pipeline did not see it), or the events do not surface an expectation failure (the bad file had no null order_id rows, or the expect_or_fail decorator was dropped).

</details>

<details><summary>Hint 2 (direction)</summary>

The ndjson payload looks like `"\n".join(json.dumps(row) for row in BAD_ORDERS) + "\n"`. Encode as utf-8 bytes and pass through `io.BytesIO`. After upload, `w.pipelines.start_update(PIPELINE_ID, full_refresh=False).update_id`, then poll `w.pipelines.get_update(PIPELINE_ID, update_id)` until the state is FAILED.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
ndjson = "\n".join(json.dumps(row) for row in BAD_ORDERS) + "\n"
w.files.upload(
    file_path=f"{SOURCE_VOLUME_PATH}/bad-orders.json",
    contents=io.BytesIO(ndjson.encode("utf-8")),
    overwrite=True,
)

upd = w.pipelines.start_update(PIPELINE_ID, full_refresh=False)
update_id = upd.update_id

deadline = time.time() + 300
final_state = None
while time.time() < deadline:
    info = w.pipelines.get_update(PIPELINE_ID, update_id)
    state = (info.update.state.value if info.update and info.update.state else "")
    state = state.upper() if state else ""
    if state in ("COMPLETED", "FAILED", "CANCELED"):
        final_state = state
        break
    time.sleep(8)

print(f"Bad-run final state: {final_state}")
```

</details>

**Wallet check.** Still at $0.00. Two pipeline runs total against tiny inputs. If you re-ran the failing pipeline four or five times debugging, you have used a meaningful slice of today's quota; cleanup at the end of the lab releases the pipeline.

## Task 4: Verify pipeline.py is the only source of truth

The validator does two things:

1. Re-fetches the pipeline and asserts `libraries` has exactly one entry pointing at `pipeline.py`. No second library, no inline notebook.
2. Reads pipeline.py from the workspace and asserts the file does NOT contain `saveAsTable` or `write.toTable`. Direct DataFrame writes inside `@dlt.table` functions defeat the declarative contract; the decorators handle materialization.

You should not need to write any new code for this task if the Task 2 build was clean. If the validator complains, the fix is in pipeline.py source.

In [ ]:
# NBVAL_SKIP
# Task 4: No new code; the checkpoint reads the workspace pipeline.py and
# the pipeline's libraries field.

if PIPELINE_ID:
    info = w.pipelines.get(PIPELINE_ID)
    print(f"Pipeline libraries: {len(info.spec.libraries or [])}")
    for lib in (info.spec.libraries or []):
        if lib.file:
            print(f"  file: {lib.file.path}")
        if lib.notebook:
            print(f"  notebook: {lib.notebook.path}")
else:
    print("PIPELINE_ID is None; re-run Task 2.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: libraries list has exactly one entry; pipeline.py text does
# NOT contain saveAsTable or write.toTable substrings (the decorators are
# the only declared writers).


def checkpoint_4(session):
    try:
        if PIPELINE_ID is None:
            return CheckpointResult(
                status="fail",
                error_reason="PIPELINE_ID is None.",
            )
        info = w.pipelines.get(PIPELINE_ID)
        libs = info.spec.libraries or []
        if len(libs) != 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Pipeline has {len(libs)} libraries; expected exactly 1."
                ),
            )

        try:
            export = w.workspace.export(PIPELINE_SOURCE_PATH, format=ws_svc.ExportFormat.SOURCE)
            content = base64.b64decode(export.content or "").decode("utf-8")
        except Exception as exc:
            return CheckpointResult(
                status="fail",
                error_reason=f"Could not read pipeline.py back from workspace: {exc}",
            )
        if not content.strip():
            return CheckpointResult(
                status="fail",
                error_reason="pipeline.py is empty.",
            )
        if "saveAsTable" in content or "write.toTable" in content:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "pipeline.py contains saveAsTable or write.toTable. "
                    "Declarative pipelines materialize via @dlt.table; remove the "
                    "direct DataFrame writes."
                ),
            )
        # Sanity check the three decorators are present.
        for needle in ("@dlt.table", "@dlt.expect_or_drop", "@dlt.expect_or_fail"):
            if needle not in content:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"pipeline.py is missing {needle!r}. The pipeline source "
                        f"must declare three tables with the two expectation tiers."
                    ),
                )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint reads pipeline.py from the workspace and inspects the pipeline's libraries field. If it complains about saveAsTable, you added a direct write inside an `@dlt.table` function (anti-pattern). If it complains about the expectation decorators, your pipeline.py is the wrong file (something overwrote the workspace path).

</details>

<details><summary>Hint 2 (direction)</summary>

No code to write. If the checkpoint complains, the fix is to re-import the `PIPELINE_SOURCE` string into `PIPELINE_SOURCE_PATH` via `w.workspace.import_` with `overwrite=True`. The canonical pipeline.py is the `PIPELINE_SOURCE` constant from the imports cell.

</details>

<details><summary>Hint 3 (spoiler)</summary>

If you need to re-write pipeline.py:

```python
import base64
from databricks.sdk.service.workspace import ImportFormat, Language

w.workspace.import_(
    path=PIPELINE_SOURCE_PATH,
    format=ImportFormat.SOURCE,
    language=Language.PYTHON,
    content=base64.b64encode(PIPELINE_SOURCE.encode("utf-8")).decode("utf-8"),
    overwrite=True,
)
```

</details>

**Wallet check.** Still at $0.00. The validator runs a workspace export and a pipelines get; both are metadata operations. Pipeline runs are the only thing on serverless and they finished in Task 3.

## Cleanup

Time to tear it all down. The cell below stops the pipeline (if running), deletes it, drops the three managed tables, clears the volume contents and drops the volume, deletes the workspace pipeline.py path, then drops the schema with `CASCADE`. The pipeline is flagged critical because a running pipeline consumes daily quota; the stop call is idempotent.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST. The pipeline is
# critical=True; it goes first.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_resources = [r for r in CLEANUP_MANIFEST if r.id not in failed_ids and getattr(r, "critical", False)]
standard_resources = [r for r in CLEANUP_MANIFEST if r.id not in failed_ids and not getattr(r, "critical", False)]
critical_destroyed = len(critical_resources)
standard_destroyed = len(standard_resources)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: $0.00.** One Lakeflow Spark Declarative Pipeline, two runs (one clean, one deliberately failed), three managed tables built declaratively, and one volume with four JSON files. Free Edition swallowed the daily quota share. The expensive thing this session would have prevented is the brittle three-cell notebook silently shipping a wrong daily total to the CFO.

## Reflection

These are not graded. They are for you.

1. Walk through what a Lakeflow Spark Declarative Pipeline does when a row violates an `@dlt.expect_or_drop` expectation vs an `@dlt.expect_or_fail` expectation. Name the row-level behavior and the update-level behavior for each. Why does Databricks ship three severity tiers (`expect`, `expect_or_drop`, `expect_or_fail`) instead of two?

2. Your team is choosing between a declarative pipeline and a notebook-based Lakeflow Jobs DAG of three tasks (ingest, transform, validate) for the same medallion workload. Sketch the trade-offs: code shape, observability, recovery on failure, refactor effort. When does each pattern win?

## Exam-Style Practice

**Question 1 (Domain 3 / 4, expectation severity tiers):**

A Lakeflow Spark Declarative Pipeline defines a silver table with the following decorators:

```python
@dlt.table
@dlt.expect("non_negative_price", "price >= 0")
@dlt.expect_or_drop("non_null_customer", "customer_id IS NOT NULL")
@dlt.expect_or_fail("unique_order_id", "order_id IS NOT NULL")
def silver_orders():
    return spark.read.table("bronze_orders")
```

A bronze row arrives with `price = -5`, a non-null `customer_id`, and a non-null `order_id`. What happens to that row in the silver table?

A. The row is dropped because `expect_or_drop` semantics apply to every expectation.
B. The row is included in silver, the violation is recorded in the pipeline event log, and the update continues.
C. The entire update fails because `expect_or_fail` is present anywhere in the table's decorators.
D. The row is included in silver but the `price` column is null-coalesced to 0 by Lakeflow.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: `dlt.expect` (without `_or_drop` or `_or_fail`) records the violation in the event log but does NOT drop the row. Only `expect_or_drop` drops violators.
- B is correct: `dlt.expect` is the lowest severity tier. The row is kept in silver, the violation count increments on the `non_negative_price` expectation, and the update continues. The pipeline event log captures the violation for downstream observability.
- C is wrong: `expect_or_fail` fails the update only when ITS predicate fails. In this scenario `order_id` is non-null so the fail-expectation passes. The other expectations are independent.
- D is wrong: Lakeflow does not silently coerce values. There is no null-coalescing of `price` based on the expectation.

</details>

**Question 2 (Domain 4, pipeline trigger modes):**

A team builds a Lakeflow Spark Declarative Pipeline to process orders. The orders arrive in batches every six hours and the team wants the pipeline to run on a fixed six-hour cadence. Which configuration is best?

A. Set the pipeline to continuous mode so it processes orders in real time as they arrive.
B. Set the pipeline to triggered mode and schedule it from a Lakeflow Job with a six-hour cron trigger.
C. Set the pipeline to continuous mode and add a `time.sleep(21600)` call at the top of the pipeline source file.
D. Set the pipeline to triggered mode without a schedule and rely on manual trigger from the team's on-call.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: continuous mode runs the pipeline continuously and bills for compute the whole time. For a six-hour batch workload, continuous wastes spend.
- B is correct: triggered mode plus a Lakeflow Job with a six-hour cron is the documented Databricks pattern for predictable-cadence batch pipelines. The pipeline runs only when the job triggers it; compute spins down between runs.
- C is wrong: `time.sleep` in a pipeline source file is an anti-pattern; continuous mode does not honor sleeps and the compute meter keeps running.
- D is wrong: relying on manual trigger removes the cadence guarantee and creates a single-person dependency on the on-call.

</details>